# 04 — Exp 2 & Exp 3: PageMantle End-to-End & Online `DB-CSE` Skill Evolution

1. Enforce **Invariant #8** (`verify_split_isolation`): `pm_official_8` & `pm_eval_32` NEVER enter `pm_online_stream_24`.
2. Evaluate pre-evolution baselines (`B0 Direct`, `B1 Single Gate`, `B2 ProACT-Enterprise`, `DIAL-Skill-v0` on `Luna` & `Terra`).
3. Run 3 sequential online waves (`pm_online_stream_24`) comparing `Static-v0`, `Reflexion-Append`, `AWM-Workflow`, `DB-CSE w/o Escrow`, and `DB-CSE Full` (tracking per-wave scores & prompt-token growth).
4. Evaluate post-evolution generalization strictly on held-out `pm_official_8` + `pm_eval_32`.

In [1]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent DRIVE_ROOT: /content/drive/MyDrive/when2speak_research


In [2]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

# Ensure `src/` is immediately importable in a fresh Colab kernel without restarting Python
_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


Already up to date.
/content/when2speak
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done


In [3]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


Secret lookup diagnostics: {
  "jev": {
    "status": "found",
    "source": "drive:JEV_API_KEY.txt",
    "masked": "apik...c3aa"
  },
  "openai": {
    "status": "found",
    "source": "drive:OPENAI_KEY.txt",
    "masked": "sk-p...yN8A"
  }
}
Resolved models: {'timestamp_utc': '2026-11-20T13:25:11.705978+00:00', 'luna': 'gpt-4.1-mini', 'terra': 'gpt-4.1', 'sol': 'gpt-6.1-sol', 'jev_model': 'jev-latest', 'jev_healthy': True, 'triage_backend': 'jev', 'available_openai_models_count': 99, 'notes': ["Probe failed for luna candidate 'gpt-6-luna': BadRequestError", "Probe failed for luna candidate 'gpt-5.6-luna': BadRequestError", "Probe failed for terra candidate 'gpt-6.1-terra': BadRequestError", "Probe failed for terra candidate 'gpt-5.6-terra': BadRequestError"]}


## Step 1: Load Splits, Enforce Split Isolation & Configure Mode

In [4]:
from dial_skill.benchmarks.build_pagemantle_ext import verify_split_isolation
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.schemas import PageMantleTask
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "full"  # Change to "full" (all 64 tasks: 40 held-out eval + 24 stream tasks across 3 waves) for paper Tables 2 & 3!
PM_DIR = DRIVE_ROOT / "datasets" / "pagemantle_ext"
RUN_MAIN = DRIVE_ROOT / "runs" / "main_pagemantle" / MODE
RUN_ONLINE = DRIVE_ROOT / "runs" / "online_dbcse_waves" / MODE

def _load_pm(name: str):
    p = PM_DIR / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_stream = _load_pm("pm_online_stream_24")
pm_eval = _load_pm("pm_eval_32")
verify_split_isolation(pm_official, pm_stream, pm_eval, strict_counts=True)

if MODE == "pilot":
    pm_eval_sub = [t for t in pm_eval if t.expected_action == "INTERVENE"][:4] + [
        t for t in pm_eval if t.expected_action == "SILENT"
    ][:4]
    eval_tasks = pm_official + pm_eval_sub
    stream_tasks = [t for t in pm_stream if t.wave == 1]
else:
    eval_tasks = pm_official + pm_eval
    stream_tasks = pm_stream

print(f"MODE={MODE} | Held-out eval tasks={len(eval_tasks)} (traps={sum(1 for t in eval_tasks if t.expected_action=='SILENT')}) | Online stream tasks={len(stream_tasks)}")
print("Estimated cost:", json.dumps(estimate_run_cost_usd(mode=MODE, include_pagemantle=True), indent=2))
save_run_metadata(RUN_MAIN, {"mode": MODE, "eval_count": len(eval_tasks), "stream_count": len(stream_tasks)})


MODE=full | Held-out eval tasks=40 (traps=10) | Online stream tasks=24
Estimated cost: {
  "mode": "full",
  "estimated_api_calls": 48607,
  "estimated_when2speak_usd": 47.71,
  "estimated_pagemantle_usd": 10.75,
  "estimated_ablations_usd": 0.0,
  "estimated_total_usd": 58.46
}


PosixPath('/content/drive/MyDrive/when2speak_research/runs/main_pagemantle/full/run_config.json')

## Step 2: Exp 2 — Pre-Evolution Evaluation on Held-Out PageMantle

In [5]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.baselines.proact import ProACTBaseline
from dial_skill.baselines.single_gate import SingleGateBaseline
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_pagemantle_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("terra", MODELS_CFG, RESOLVED_MODELS)
backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=backend)

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
env.ensure_extracted()
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

pre_agents = {
    "B0_direct_luna": DirectChatBaseline(openai_client=oai, model=luna_model),
    "B0_direct_terra": DirectChatBaseline(openai_client=oai, model=terra_model),
    "B1_single_gate": SingleGateBaseline(openai_client=oai, gate_model=luna_model, executor_model=terra_model),
    "B2_proact_enterprise": ProACTBaseline(openai_client=oai, gate_model=luna_model, executor_model=terra_model),
    "DIAL_Skill_v0_luna": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model,
        skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_v0_luna",
    ),
    "DIAL_Skill_v0_terra": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
        skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_v0_terra",
    ),
}

pre_metrics = {}
for name, agent in pre_agents.items():
    eps = await run_pagemantle_eval(
        agent=agent, tasks=eval_tasks, env=env, output_jsonl=RUN_MAIN / f"preds_{name}.jsonl", resume=True
    )
    scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(eval_tasks, eps)]
    (RUN_MAIN / f"judge_{name}.jsonl").write_text("\n".join(s.model_dump_json() for s in scores) + "\n", encoding="utf-8")
    pre_metrics[name] = compute_pagemantle_metrics(eval_tasks, eps, scores)
    m = pre_metrics[name]
    print(f"[{name:22s}] Overall={m['overall_score']:.2f} | SilAcc={m['silence_accuracy']:.2f} | ActRec={m['action_recall']:.2f} | Aud={m['audience_and_location']:.2f} | Cost=${m['cost_per_task_usd']:.4f}")

(RUN_MAIN / "pre_evolution_metrics.json").write_text(json.dumps(pre_metrics, indent=2), encoding="utf-8")


PM[preds_B0_direct_luna]:   0%|          | 0/40 [00:00<?, ?it/s]

[B0_direct_luna        ] Overall=7.49 | SilAcc=0.90 | ActRec=0.90 | Aud=6.90 | Cost=$0.0016


PM[preds_B0_direct_terra]:   0%|          | 0/40 [00:00<?, ?it/s]

[B0_direct_terra       ] Overall=4.90 | SilAcc=1.00 | ActRec=0.40 | Aud=4.60 | Cost=$0.0045


PM[preds_B1_single_gate]:   0%|          | 0/40 [00:00<?, ?it/s]

[B1_single_gate        ] Overall=5.09 | SilAcc=1.00 | ActRec=0.47 | Aud=4.59 | Cost=$0.0035


... (3 outputs left out)

[DIAL_Skill_v0_luna    ] Overall=8.32 | SilAcc=1.00 | ActRec=0.97 | Aud=8.38 | Cost=$0.0073


PM[preds_DIAL_Skill_v0_terra]:   0%|          | 0/40 [00:00<?, ?it/s]

[DIAL_Skill_v0_terra   ] Overall=8.61 | SilAcc=1.00 | ActRec=0.97 | Aud=8.41 | Cost=$0.0059


4645

## Step 3: Exp 3 — Online Evolution (`Static-v0` vs `Reflexion-Append` vs `AWM-Workflow` vs `DB-CSE w/o Escrow` vs `DB-CSE Full`)

In [6]:
from dial_skill.eval.runners import run_online_evolution_stream
from dial_skill.evolution.baselines import AWMWorkflowUpdater, ReflexionAppendUpdater
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.evolution.feedback import extract_feedback_from_pagemantle
from dial_skill.schemas import EpisodeResult
from dial_skill.utils.checkpoint import load_jsonl

# 1. DB-CSE Full (with Counterfactual Replay Escrow Gate & hard O(1) Cap)
dbcse_full_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
    skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_DBCSE_Full",
)
dbcse_full_updater = DBCSEUpdater(
    skill_store=dbcse_full_agent.skill_store, openai_client=oai, critic_model=terra_model,
    cfg=DIAL_CFG, log_path=RUN_ONLINE / "dbcse_full" / "skill_evolution_log.jsonl",
)
full_stream_summary = await run_online_evolution_stream(
    agent=dbcse_full_agent, updater=dbcse_full_updater, stream_tasks=stream_tasks,
    env=env, run_dir=RUN_ONLINE / "dbcse_full", judge=judge,
)

# 2. DB-CSE w/o Escrow Gate ablation
no_escrow_cfg = load_dial_skill_config(overrides={"evolution": {"enable_escrow_gate": False}})
dbcse_noesc_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
    skill_store=SkillStore.from_json_file(), cfg=no_escrow_cfg, method_name="DIAL_Skill_DBCSE_NoEscrow",
)
dbcse_noesc_updater = DBCSEUpdater(
    skill_store=dbcse_noesc_agent.skill_store, openai_client=oai, critic_model=terra_model,
    cfg=no_escrow_cfg, log_path=RUN_ONLINE / "dbcse_no_escrow" / "skill_evolution_log.jsonl",
)
noesc_stream_summary = await run_online_evolution_stream(
    agent=dbcse_noesc_agent, updater=dbcse_noesc_updater, stream_tasks=stream_tasks,
    env=env, run_dir=RUN_ONLINE / "dbcse_no_escrow", judge=judge,
)

# 3. Static-v0, Reflexion-Append, and AWM-Workflow stream comparators (for Tab 3, Fig 2 & Fig 5 O(1) vs O(T) curve!)
static_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
    skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="Static_v0",
)
static_stream_eps = await run_pagemantle_eval(
    agent=static_agent, tasks=stream_tasks, env=env, output_jsonl=RUN_ONLINE / "static_v0_stream.jsonl", resume=True
)
static_stream_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(stream_tasks, static_stream_eps)]

refl_updater = ReflexionAppendUpdater(openai_client=oai, model=terra_model)
awm_updater = AWMWorkflowUpdater(openai_client=oai, model=terra_model)
base_store_tokens = SkillStore.from_json_file().estimate_serialized_tokens()
refl_token_curve = [base_store_tokens]
awm_token_curve = [base_store_tokens]
for task, ep, j_score in zip(stream_tasks, static_stream_eps, static_stream_scores):
    fb = extract_feedback_from_pagemantle(ep, task, j_score)
    await refl_updater.process_episode(ep, fb)
    await awm_updater.process_episode(ep, fb)
    refl_token_curve.append(base_store_tokens + refl_updater.estimate_memory_tokens())
    awm_token_curve.append(base_store_tokens + awm_updater.estimate_memory_tokens())

refl_updater.save(RUN_ONLINE / "reflexion_memory.json")
awm_updater.save(RUN_ONLINE / "awm_workflows.json")

waves = sorted({t.wave or 1 for t in stream_tasks})
static_wave_scores = []
for wid in waves:
    w_t = [t for t in stream_tasks if (t.wave or 1) == wid]
    w_e = [e for t, e in zip(stream_tasks, static_stream_eps) if (t.wave or 1) == wid]
    w_s = [s for t, s in zip(stream_tasks, static_stream_scores) if (t.wave or 1) == wid]
    static_wave_scores.append(compute_pagemantle_metrics(w_t, w_e, w_s, n_bootstrap=200)["overall_score"])

# 4. Evaluate evolved agents strictly on held-out eval_tasks (zero leakage!)
post_metrics = {}
for name, agent in [
    ("DIAL_Skill_Evolved_Full", dbcse_full_agent),
    ("DIAL_Skill_Evolved_NoEscrow", dbcse_noesc_agent),
]:
    eps = await run_pagemantle_eval(
        agent=agent, tasks=eval_tasks, env=env, output_jsonl=RUN_MAIN / f"preds_{name}.jsonl", resume=False
    )
    scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(eval_tasks, eps)]
    (RUN_MAIN / f"judge_{name}.jsonl").write_text("\n".join(s.model_dump_json() for s in scores) + "\n", encoding="utf-8")
    post_metrics[name] = compute_pagemantle_metrics(eval_tasks, eps, scores)
    m = post_metrics[name]
    print(f"[{name:28s} Held-Out] Overall={m['overall_score']:.2f} | SilAcc={m['silence_accuracy']:.2f} | ActRec={m['action_recall']:.2f}")

(RUN_MAIN / "post_evolution_metrics.json").write_text(json.dumps(post_metrics, indent=2), encoding="utf-8")

online_comparison = {
    "waves": waves,
    "base_store_tokens": base_store_tokens,
    "static_v0": {
        "wave_scores": static_wave_scores,
        "heldout_score": pre_metrics["DIAL_Skill_v0_terra"]["overall_score"],
    },
    "dbcse_full": {
        "wave_scores": [w["overall_score"] for w in full_stream_summary["waves"]],
        "heldout_score": post_metrics["DIAL_Skill_Evolved_Full"]["overall_score"],
        "stats": full_stream_summary["dbcse_stats"],
        "token_curve": [p["skill_store_tokens"] for p in full_stream_summary["prompt_token_curve"]],
    },
    "dbcse_no_escrow": {
        "wave_scores": [w["overall_score"] for w in noesc_stream_summary["waves"]],
        "heldout_score": post_metrics["DIAL_Skill_Evolved_NoEscrow"]["overall_score"],
        "stats": noesc_stream_summary["dbcse_stats"],
        "token_curve": [p["skill_store_tokens"] for p in noesc_stream_summary["prompt_token_curve"]],
    },
    "reflexion_append": {"token_curve": refl_token_curve, "n_reflections": len(refl_updater.reflections)},
    "awm_workflow": {"token_curve": awm_token_curve, "n_workflows": len(awm_updater.workflows)},
}
(RUN_ONLINE / "online_comparison_summary.json").write_text(json.dumps(online_comparison, indent=2), encoding="utf-8")
print("Saved online_comparison_summary.json | DB-CSE Full stats:", full_stream_summary["dbcse_stats"])
print(f"Token growth (step 0 -> {len(stream_tasks)}): DB-CSE={online_comparison['dbcse_full']['token_curve'][0]}->{online_comparison['dbcse_full']['token_curve'][-1]} | Reflexion={refl_token_curve[0]}->{refl_token_curve[-1]} | AWM={awm_token_curve[0]}->{awm_token_curve[-1]}")


PM[static_v0_stream]:   0%|          | 0/24 [00:00<?, ?it/s]

PM[preds_DIAL_Skill_Evolved_Full]:   0%|          | 0/40 [00:00<?, ?it/s]

[DIAL_Skill_Evolved_Full      Held-Out] Overall=8.59 | SilAcc=1.00 | ActRec=0.97


PM[preds_DIAL_Skill_Evolved_NoEscrow]:   0%|          | 0/40 [00:00<?, ?it/s]

[DIAL_Skill_Evolved_NoEscrow  Held-Out] Overall=8.60 | SilAcc=1.00 | ActRec=0.97
Saved online_comparison_summary.json | DB-CSE Full stats: {'episodes_processed': 24, 'numeric_updates': 18, 'patches_proposed': 4, 'patches_rejected_guardrail': 0, 'patches_blocked_escrow': 0, 'patches_promoted': 4, 'pruning_events': 0}
Token growth (step 0 -> 24): DB-CSE=2522->2859 | Reflexion=2522->2987 | AWM=2522->5706
